# Notebook 07 — Graph-Enhanced Model and Evaluation

## Purpose
Test the project's central question directly: does adding graph-derived 
structural features (in-degree, out-degree) close the precision gap found 
in notebook 06's honest baseline (P=0.28 without the simulator artefact 
`errorBalanceOrig`)?

## Features
- Honest tabular features (notebook 06, excluding errorBalanceOrig)
- Graph features: in-degree, out-degree (from notebook 02) — PageRank and 
  embeddings were tested and excluded in notebooks 04-05 with evidence

## Comparison
Three models evaluated head to head:
1. Full baseline (with errorBalanceOrig) — notebook 06, for reference
2. Honest baseline (without errorBalanceOrig) — notebook 06
3. Honest baseline + graph features — this notebook

In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pickle
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_score, recall_score, f1_score,
                               classification_report, confusion_matrix,
                               average_precision_score)

# Load the filtered transactions
df = pd.read_csv('../data/processed/filtered_transactions.csv')

# Load the saved graph (already has in_degree, out_degree as node attributes from notebook 02)
with open('../data/processed/transaction_graph.pkl', 'rb') as f:
    G = pickle.load(f)

print(f"Transactions: {df.shape}")
print(f"Graph nodes: {G.number_of_nodes():,}")

Transactions: (2770393, 11)
Graph nodes: 3,277,489


In [5]:
# Extract in-degree and out-degree for every node from the graph
in_degree_dict = dict(G.in_degree())
out_degree_dict = dict(G.out_degree())

# Map these onto each transaction — sender's out-degree, receiver's in-degree
# (and also sender's in-degree, receiver's out-degree — capturing both roles)
df['orig_out_degree'] = df['nameOrig'].map(out_degree_dict)
df['orig_in_degree'] = df['nameOrig'].map(in_degree_dict)
df['dest_in_degree'] = df['nameDest'].map(in_degree_dict)
df['dest_out_degree'] = df['nameDest'].map(out_degree_dict)

print(df[['orig_out_degree', 'orig_in_degree', 'dest_in_degree', 'dest_out_degree']].describe())

       orig_out_degree  orig_in_degree  dest_in_degree  dest_out_degree
count     2.770393e+06    2.770393e+06    2.770393e+06     2.770393e+06
mean      1.001286e+00    1.399440e-03    1.158245e+01     1.399440e-03
std       3.593470e-02    1.283906e-01    8.666530e+00     3.740222e-02
min       1.000000e+00    0.000000e+00    1.000000e+00     0.000000e+00
25%       1.000000e+00    0.000000e+00    5.000000e+00     0.000000e+00
50%       1.000000e+00    0.000000e+00    9.000000e+00     0.000000e+00
75%       1.000000e+00    0.000000e+00    1.600000e+01     0.000000e+00
max       3.000000e+00    4.200000e+01    7.500000e+01     2.000000e+00


In [6]:
# Engineer the same honest tabular features as the fair baseline (notebook 06)
df['type_TRANSFER'] = (df['type'] == 'TRANSFER').astype(int)
df['type_CASH_OUT'] = (df['type'] == 'CASH_OUT').astype(int)

# Feature set: honest tabular features + graph features
feature_cols_graph = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig', 
    'oldbalanceDest', 'newbalanceDest',
    'type_TRANSFER', 'type_CASH_OUT',
    'orig_out_degree', 'orig_in_degree', 
    'dest_in_degree', 'dest_out_degree'
]

X_graph = df[feature_cols_graph]
y = df['isFraud']

X_train_g, X_test_g, y_train_g, y_test_g = train_test_split(
    X_graph, y, test_size=0.2, random_state=42, stratify=y
)

scale_pos_weight_g = (y_train_g == 0).sum() / (y_train_g == 1).sum()

model_graph = xgb.XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_g, random_state=42, eval_metric='aucpr'
)
model_graph.fit(X_train_g, y_train_g)

y_pred_g = model_graph.predict(X_test_g)
y_pred_proba_g = model_graph.predict_proba(X_test_g)[:, 1]

print(f"Precision: {precision_score(y_test_g, y_pred_g):.4f}")
print(f"Recall:    {recall_score(y_test_g, y_pred_g):.4f}")
print(f"F1 Score:  {f1_score(y_test_g, y_pred_g):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test_g, y_pred_proba_g):.4f}")

Precision: 0.2661
Recall:    0.9969
F1 Score:  0.4201
PR-AUC:    0.9388


Graph features (in-degree/out-degree) added to honest baseline:
P=0.2661, R=0.9969, F1=0.4201, PR-AUC=0.9388
vs honest baseline alone: P=0.2808, R=0.9951, F1=0.4380, PR-AUC=0.9398
Result: no improvement — marginally worse on precision/F1/PR-AUC
Consistent with notebooks 04/05: this network lacks the multi-hop structure 
graph methods exploit. XGBoost's tree splits likely already approximate 
degree-equivalent signal via amount/balance correlations.
FINAL VERDICT: across PageRank, embeddings, and explicit degree features, 
no graph technique improved on the tabular baseline for this dataset. This 
is a defensible, evidence-based answer to the project's central question — 
not a failure of graph methods generally, but a specific structural finding 
about this network.
